# Neurosolit SFT on Colab (Qwen2.5-7B, QLoRA, Merged FP16 for Spaces)

Runtime > Change runtime type > **T4 GPU**. Standard RAM works completely fine with the auto device map.

Run the cells in order:
1. **Cell 1**: GPU setup and version pinning.
2. **Cell 2 & 3**: Upload `neurosolit_colab.zip` and extract.
3. **Cell 4**: Hugging Face Login.
4. **Cell 5 & 6**: Train 7B QLoRA adapter & evaluate benchmark gates.
5. **Cell 7**: Merge LoRA adapter into full FP16 base model.
6. **Cell 8**: Push the merged 7B model to `Scadiva/nd-solitude-7b-merged` for Spaces!

In [ ]:
# 1. GPU check + pinned versions
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip uninstall -y torchao
!pip install -q -U "trl==0.19.1" "transformers==4.53.3" "peft==0.21.2" bitsandbytes accelerate datasets huggingface_hub


In [ ]:
# 2. Upload neurosolit_colab.zip
from google.colab import files
import zipfile, os, shutil

up = files.upload()
name = next(iter(up))
os.makedirs('/content/proj', exist_ok=True)
zipfile.ZipFile(name).extractall('/content/proj')
os.chdir('/content/proj')
os.makedirs('training', exist_ok=True)
os.makedirs('data', exist_ok=True)
for f in list(os.listdir('.')):
    if f == 'train_sft.py':
        shutil.move(f, 'training/train_sft.py')
    elif f.endswith('.jsonl'):
        shutil.move(f, 'data/' + f)
print('Project root:', sorted(os.listdir('.')))
print('Data batches:', sorted(os.listdir('data')))


In [ ]:
# 3. Sanity check: all files present
need = ['run_pipeline.py', 'eval_models.py', 'training/train_sft.py',
        'data/pairs_v4_handwritten.jsonl', 'data/new_reviewed_batch1.jsonl',
        'data/new_reviewed_batch2.jsonl', 'data/new_reviewed_batch3.jsonl',
        'data/new_reviewed_batch4.jsonl']
missing = [p for p in need if not os.path.exists(p)]
print('MISSING:' if missing else 'All files present. Ready to train!', missing)


In [ ]:
# 4. Hugging Face login
from huggingface_hub import login
try:
    from google.colab import userdata
    login(userdata.get('HF_TOKEN'))
except Exception:
    import getpass
    login(getpass.getpass('HF token: '))


In [ ]:
# 5. Filter data -> train 7B QLoRA -> evaluate. Does NOT push.
# First run downloads ~15 GB base model; training takes roughly 15-25 min on a T4 GPU.
!python run_pipeline.py \
  --data data/pairs_v4_handwritten.jsonl \
  --data data/new_reviewed_batch1.jsonl \
  --data data/new_reviewed_batch2.jsonl \
  --data data/new_reviewed_batch3.jsonl \
  --data data/new_reviewed_batch4.jsonl \
  --min-rows 60 --epochs 3 --max-length 320 \
  --base-model Qwen/Qwen2.5-7B-Instruct --no-merge \
  --hf-repo Scadiva/nd-solitude-7b-adapter


In [ ]:
# 6. Read sample replies, including novel off-pattern prompts
!python eval_models.py --base Qwen/Qwen2.5-7B-Instruct --adapter pipeline_out/adapter --load-4bit \
  --temperature 0.4 --samples 3 --show 2 \
  --extra-prompt "What is the capital of France?" \
  --extra-prompt "I've been staring at the wall for four hours and cannot move my arms."


In [ ]:
# 7. Merge the LoRA adapter with Qwen2.5-7B base model into FP16
# Add 10GB virtual swap memory so CPU has 22GB+ RAM for clean, meta-free merging
!fallocate -l 10G /swapfile 2>/dev/null && chmod 600 /swapfile && mkswap /swapfile 2>/dev/null && swapon /swapfile 2>/dev/null || true

import torch, gc
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

base_model = 'Qwen/Qwen2.5-7B-Instruct'
adapter_dir = 'pipeline_out/adapter'
merged_dir = 'pipeline_out/merged'

print('1. Loading base 7B model on CPU (with swap headroom)...')
base_fp16 = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype=torch.float16,
    device_map='cpu',
    low_cpu_mem_usage=True
)
tokenizer = AutoTokenizer.from_pretrained(base_model)

print('2. Merging LoRA adapter weights across all 28 layers...')
merged = PeftModel.from_pretrained(base_fp16, adapter_dir).merge_and_unload()

print(f'3. Saving merged model to {merged_dir} (sharded in 4GB chunks)...')
merged.save_pretrained(merged_dir, max_shard_size='4GB')
tokenizer.save_pretrained(merged_dir)
print('Merge complete! pipeline_out/merged is ready for upload.')


In [ ]:
# 8. Push the merged 7B model to Hugging Face Hub (Public for Spaces)
from huggingface_hub import HfApi
api = HfApi()
repo_id = 'Scadiva/nd-solitude-7b-merged'

print(f'Creating repo {repo_id}...')
api.create_repo(repo_id=repo_id, exist_ok=True, private=False)

print(f'Uploading 7B merged weights from pipeline_out/merged...')
api.upload_folder(
    folder_path='pipeline_out/merged',
    repo_id=repo_id,
    commit_message='Add fine-tuned Qwen2.5-7B low-demand solitude model'
)
print(f'Done! Model is live at https://huggingface.co/{repo_id}')


In [ ]:
# 9. Optional: Download a backup zip of the adapter
!zip -qr adapter_7b.zip pipeline_out/adapter
from google.colab import files
files.download('adapter_7b.zip')
